In [1]:
from google.colab import drive
drive.mount('/content/drive')
# Create a folder in your Drive to store project files (runs only once)
!mkdir -p /content/drive/MyDrive/milestone4
PROJECT_DIR = "/content/drive/MyDrive/milestone4"
PROJECT_DIR


Mounted at /content/drive


'/content/drive/MyDrive/milestone4'

In [2]:
# Install if missing
!pip install --quiet ipywidgets pandas scikit-learn
# (Colab already has many libs; this ensures ipywidgets is available)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.2 MB/s eta 0:00:00


In [3]:
import sqlite3, hashlib
from datetime import datetime
import os

PROJECT_DIR = "/content/drive/MyDrive/milestone4"
DB_PATH = os.path.join(PROJECT_DIR, 'study_logs.db')

def get_conn():
    return sqlite3.connect(DB_PATH)

# create folder if not exists
os.makedirs(PROJECT_DIR, exist_ok=True)

with get_conn() as conn:
    cur = conn.cursor()
    cur.execute('''CREATE TABLE IF NOT EXISTS students (
        student_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        email TEXT UNIQUE)''')
    cur.execute('''CREATE TABLE IF NOT EXISTS study_logs (
        log_id INTEGER PRIMARY KEY AUTOINCREMENT,
        student_id INTEGER,
        timestamp TEXT,
        subject TEXT,
        duration_minutes INTEGER,
        notes TEXT,
        FOREIGN KEY(student_id) REFERENCES students(student_id))''')
    cur.execute('''CREATE TABLE IF NOT EXISTS admin (username TEXT PRIMARY KEY, password_hash TEXT)''')
    conn.commit()

print("DB created at:", DB_PATH)


DB created at: /content/drive/MyDrive/milestone4/study_logs.db


In [4]:
def create_admin(username, password):
    pwd_hash = hashlib.sha256(password.encode()).hexdigest()
    with get_conn() as conn:
        conn.execute('INSERT OR REPLACE INTO admin (username, password_hash) VALUES (?,?)', (username, pwd_hash))
        conn.commit()

# Run once to set admin — change password to something you choose
create_admin('admin','yourStrongPassword')
print("Admin created: username='admin'  (remember your password)")


Admin created: username='admin'  (remember your password)


In [5]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime

# Widgets
name_input = widgets.Text(description='Name:')
email_input = widgets.Text(description='Email:')
subject_input = widgets.Text(description='Subject:')
duration_input = widgets.IntText(description='Minutes:')
notes_input = widgets.Textarea(description='Notes:')
save_btn = widgets.Button(description='Save Study Log')
status_out = widgets.Output()
student_select = widgets.Dropdown(options=['-- new --'], description='Student:')

def refresh_student_select():
    with get_conn() as conn:
        df = pd.read_sql('SELECT student_id, name FROM students', conn)
    opts = ['-- new --'] + [f"{r.student_id}: {r.name}" for r in df.itertuples()]
    student_select.options = opts

def on_save_clicked(b):
    with status_out:
        clear_output()
        name = name_input.value.strip(); email = email_input.value.strip() or None
        subject = subject_input.value.strip(); duration = int(duration_input.value or 0)
        if not name or duration <= 0 or not subject:
            print('Enter name, subject and a positive duration.')
            return
        with get_conn() as conn:
            cur = conn.cursor()
            if student_select.value and student_select.value != '-- new --':
                sid = int(str(student_select.value).split(':')[0])
            else:
                cur.execute('INSERT OR IGNORE INTO students (name, email) VALUES (?,?)', (name, email))
                conn.commit()
                cur.execute('SELECT student_id FROM students WHERE name=? LIMIT 1', (name,))
                sid = cur.fetchone()[0]
            ts = datetime.utcnow().isoformat()
            cur.execute('INSERT INTO study_logs (student_id, timestamp, subject, duration_minutes, notes) VALUES (?,?,?,?,?)',
                        (sid, ts, subject, duration, notes_input.value.strip()))
            conn.commit()
            print('Saved study log for', name)
            refresh_student_select()

save_btn.on_click(on_save_clicked)
refresh_student_select()
display(widgets.VBox([student_select, widgets.HBox([name_input, email_input]),
                      widgets.HBox([subject_input, duration_input]), notes_input, save_btn, status_out]))


In [6]:
with get_conn() as conn:
    df = pd.read_sql('''
        SELECT l.log_id, s.student_id, s.name, l.timestamp, l.subject, l.duration_minutes, l.notes
        FROM study_logs l JOIN students s ON l.student_id = s.student_id
        ORDER BY l.log_id DESC
    ''', conn)
df.head(50)


,log_id,student_id,name,timestamp,subject,duration_minutes,notes


In [7]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import hashlib

admin_user = widgets.Text(description='Username:')
admin_pwd = widgets.Password(description='Password:')
login_btn = widgets.Button(description='Login as Admin')
admin_out = widgets.Output()
crud_out = widgets.Output()

def check_admin(username, password):
    pwd_hash = hashlib.sha256(password.encode()).hexdigest()
    with get_conn() as conn:
        row = conn.execute('SELECT password_hash FROM admin WHERE username=?', (username,)).fetchone()
        return bool(row and row[0] == pwd_hash)

def show_admin_panel():
    with crud_out:
        clear_output()
        display(pd.read_sql('SELECT * FROM students', get_conn()))
        add_name = widgets.Text(description='New name:'); add_email = widgets.Text(description='Email:')
        add_btn = widgets.Button(description='Add Student')
        upd_id = widgets.IntText(description='ID to upd:'); upd_name = widgets.Text(description='New name:')
        upd_btn = widgets.Button(description='Update'); del_id = widgets.IntText(description='ID to del:')
        del_btn = widgets.Button(description='Delete'); out2 = widgets.Output()
        def on_add(b):
            with out2:
                clear_output()
                with get_conn() as conn:
                    conn.execute('INSERT INTO students (name, email) VALUES (?,?)', (add_name.value, add_email.value or None))
                    conn.commit()
                print('Added'); show_admin_panel()
        def on_upd(b):
            with out2:
                clear_output()
                with get_conn() as conn:
                    conn.execute('UPDATE students SET name=? WHERE student_id=?', (upd_name.value, upd_id.value)); conn.commit()
                print('Updated'); show_admin_panel()
        def on_del(b):
            with out2:
                clear_output()
                with get_conn() as conn:
                    conn.execute('DELETE FROM students WHERE student_id=?', (del_id.value,)); conn.commit()
                print('Deleted'); show_admin_panel()
        add_btn.on_click(on_add); upd_btn.on_click(on_upd); del_btn.on_click(on_del)
        display(widgets.VBox([widgets.HBox([add_name, add_email, add_btn]),
                              widgets.HBox([upd_id, upd_name, upd_btn]),
                              widgets.HBox([del_id, del_btn]), out2]))

def on_login(b):
    with admin_out:
        clear_output()
        if check_admin(admin_user.value, admin_pwd.value):
            print('Login success'); show_admin_panel()
        else:
            print('Login failed')

login_btn.on_click(on_login)
display(widgets.VBox([admin_user, admin_pwd, login_btn, admin_out, crud_out]))


In [8]:
from sklearn.linear_model import LinearRegression
import numpy as np

def retrain_model():
    import pandas as pd
    with get_conn() as conn:
        df = pd.read_sql('SELECT * FROM study_logs', conn)
    if df.empty:
        print('Not enough data to retrain')
        return None
    # Toy feature: subject length -> predict duration
    X = df['subject'].fillna('').map(len).values.reshape(-1,1)
    y = df['duration_minutes'].fillna(0).values
    model = LinearRegression().fit(X, y)
    print('Retrained toy model on', len(y), 'rows')
    return model

# Example: model = retrain_model()


In [9]:
# DB already in Drive at DB_PATH. To make a local copy in Colab VM:
!cp "{DB_PATH}" /content/

# Or to download (this will generate a download link)
from google.colab import files
files.download(DB_PATH)  # triggers browser download


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
df.head(50)

,log_id,student_id,name,timestamp,subject,duration_minutes,notes
